# Train the multi-label classifier — one runnable notebook

Run this top to bottom. It fine-tunes DistilBERT on the Jigsaw toxic-comment data to predict
all six labels at once, then calibrates a threshold per label.

**Self-contained** — it does not import from `src/`, so it also runs on Colab or any fresh
machine. It writes its artifacts to the same paths the rest of the project reads, so the
later analysis notebooks pick them up automatically.

| | |
|---|---|
| **Input** | Jigsaw `train.csv` — found automatically (see below) |
| **Outputs** | `models/distilbert-multilabel/`, `models/probs/transformer_{val,test}.npy`, `results/transformer_metrics.json`, `results/thresholds_transformer.json`, `results/threshold_comparison_transformer.json` |
| **Runtime** | ~10 min on a Kaggle T4/P100 · ~45 min on an Apple M4 (MPS) · set `QUICK_TEST = True` for a ~2 min smoke run first |

### Running on Kaggle

1. **+ Add Input** → search *Jigsaw Toxic Comment Classification Challenge* → add the
   competition. The notebook finds `train.csv` under `/kaggle/input` on its own, and reads it
   whether it is plain `.csv` or `.csv.zip`.
2. **Settings → Accelerator → GPU** (T4 or P100). Works on CPU too, just slowly.
3. **Run All.** No `pip install` needed — the notebook adapts to whatever `transformers`
   version Kaggle ships, and falls back to a self-contained splitter if
   `iterative-stratification` is absent, so it works with internet turned **off**.

Outputs are written under `/kaggle/working/`, which you can download from the Output tab.

### Two things this notebook is really demonstrating

1. **Sigmoid + BCE, not softmax.** Six *independent* outputs, so a comment can be toxic AND
   obscene AND insulting at the same time. Softmax would force those to compete. See
   section 5.
2. **0.5 is the wrong threshold.** `threat` occurs in 0.30% of comments; the F1-maximizing
   cut point is nowhere near 0.5. Section 9 tunes one threshold per label on validation and
   applies them to test — the single biggest score improvement in the whole project, at no
   training cost.

## 1. Setup

Nothing to install on Kaggle or Colab — everything used here ships with both. The cell below
detects the environment, locates `train.csv`, and picks a writable output directory.

In [ ]:
import html
import json
import logging
import os
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

# Set before any tokenizer is constructed. The Rust tokenizer's internal parallelism can
# deadlock if the process later forks, which shows up as a cell that hangs with no output.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("train")

# ---------------------------------------------------------------- configuration
SEED = 42
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
N_LABELS = len(LABELS)

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256      # measured choice: truncates 6.9% of comments (17.4% of tokens). 128 would cut 19.3%.
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 128
LEARNING_RATE = 3e-5   # see the effective-batch note below
EPOCHS = 3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

# Set True for a fast end-to-end check (~3 min) before committing to the full run.
QUICK_TEST = False

# ------------------------------------------------------- environment + data discovery
IS_KAGGLE = Path("/kaggle/input").exists()

# Output root must be writable. On Kaggle only /kaggle/working is.
if IS_KAGGLE:
    ROOT = Path("/kaggle/working")
elif Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent          # running in place inside the repo
else:
    ROOT = Path.cwd()


def find_train_csv():
    """Locate Jigsaw train.csv, plain or zipped, wherever this is running.

    Kaggle mounts competition data read-only under /kaggle/input/<competition>/ and, for this
    competition, ships the CSVs zipped. pandas reads .zip directly when it holds one file, so
    both forms are accepted.
    """
    # Explicit local layout wins if present.
    for c in (ROOT / "data" / "raw" / "train.csv", Path.cwd() / "data" / "raw" / "train.csv"):
        if c.is_file():
            return c

    # Otherwise search recursively. Depth is not predictable: attaching this data as a
    # community dataset rather than the competition gives paths like
    # /kaggle/input/datasets/<user>/<dataset>/train.csv, so a fixed-depth glob misses it.
    for base in (Path("/kaggle/input"), ROOT, Path.cwd()):
        if not base.exists():
            continue
        for pattern in ("train.csv", "train.csv.zip"):
            hits = [p for p in base.rglob(pattern) if p.is_file()]
            if hits:
                # Prefer a jigsaw-looking path when several datasets are attached.
                hits.sort(key=lambda p: ("jigsaw" not in str(p).lower(), len(str(p))))
                return hits[0]

    listing = "\n  ".join(str(p) for p in sorted(Path("/kaggle/input").rglob("*.csv"))[:15]) if IS_KAGGLE else ""
    raise FileNotFoundError(
        "Could not find Jigsaw train.csv.\n"
        + ("On Kaggle: click '+ Add Input' and add the 'Jigsaw Toxic Comment Classification "
           "Challenge' data.\n" if IS_KAGGLE
           else "Place train.csv at data/raw/train.csv, or set TRAIN_CSV manually below.\n")
        + (f"CSV files currently visible under /kaggle/input:\n  {listing}" if listing else "")
    )


TRAIN_CSV = find_train_csv()

PROCESSED, MODELS, PROBS, RESULTS = (ROOT / "data" / "processed"), (ROOT / "models"), (ROOT / "models" / "probs"), (ROOT / "results")
for d in (PROCESSED, MODELS, PROBS, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

log.info("kaggle=%s | output root=%s", IS_KAGGLE, ROOT)
log.info("train data -> %s", TRAIN_CSV)


def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
log.info("device=%s | torch=%s | seed=%d | quick_test=%s", DEVICE, torch.__version__, SEED, QUICK_TEST)

# HF Trainer multiplies per_device_train_batch_size by the GPU count, so on Kaggle's T4 x2 a
# nominal batch of 32 becomes 64 and you get HALF the optimizer steps. That silently
# undertrained an earlier run (macro-F1 0.5671, and `threat` never fired at all), so make the
# effective batch visible rather than surprising.
N_GPU = torch.cuda.device_count() if DEVICE == "cuda" else 1
EFFECTIVE_BATCH = BATCH_SIZE * max(N_GPU, 1)
if DEVICE == "cuda":
    log.info("gpu=%s x%d", torch.cuda.get_device_name(0), N_GPU)
elif IS_KAGGLE:
    log.warning("no GPU detected -- enable it via Settings > Accelerator > GPU, or expect a slow run")
log.info(
    "batch: %d per device x %d device(s) = %d effective | lr=%.0e | epochs=%g",
    BATCH_SIZE, max(N_GPU, 1), EFFECTIVE_BATCH, LEARNING_RATE, EPOCHS,
)
if N_GPU > 1:
    log.warning(
        "multi-GPU: effective batch is %d, so lr=%.0e is set higher than the single-GPU 2e-5 "
        "to compensate for the halved step count", EFFECTIVE_BATCH, LEARNING_RATE,
    )

## 2. Load and clean

Cleaning is deliberately minimal — HTML-unescape and collapse whitespace, nothing more.
ALL-CAPS, `!!!!`, and repeated characters are genuine toxicity signal; TF-IDF and this
uncased model lowercase downstream anyway, so stripping it here would only lose information.

In [ ]:
_WS = re.compile(r"\s+")


def clean_text(t):
    return _WS.sub(" ", html.unescape(str(t))).strip()


df = pd.read_csv(TRAIN_CSV)
df["comment_text"] = df["comment_text"].map(clean_text)
df = df[df["comment_text"].str.len() > 0].reset_index(drop=True)

# Y is the central object: (n_samples, 6) binary. Everything downstream assumes this
# shape and this column order.
Y = df[LABELS].to_numpy(dtype=np.int8)
assert np.isin(Y, [0, 1]).all(), "train.csv labels must be 0/1"

texts = df["comment_text"].to_numpy()
log.info("%d comments | Y%s", len(df), Y.shape)

pd.DataFrame({
    "positives": Y.sum(0),
    "positive_rate_%": (100 * Y.mean(0)).round(2),
    "negatives_per_positive": (((len(Y) - Y.sum(0)) / Y.sum(0))).round(0),
}, index=LABELS)

Note the imbalance: `threat` has 333 negatives per positive. Also note that **89.8% of
comments carry no label at all** — which is why accuracy is useless here. A model that
predicts nothing scores ~99.7% accuracy on `threat`.

In [ ]:
print(f"comments with no label at all: {(Y.sum(1) == 0).mean():.1%}")
print(f"an all-zeros model would score {(Y[:, LABELS.index('threat')] == 0).mean():.2%} accuracy on `threat`,")
print(f"and {(Y.sum(1) == 0).mean():.2%} exact-match accuracy overall -- at macro-F1 of exactly 0.0.")
print("\nThat is why macro-F1 is the headline metric everywhere below.")

## 3. Train / validation / test split (70 / 15 / 15)

Multi-label-stratified, so every label's positive rate is preserved in all three splits —
it matters for `threat`, where a plain random split can leave the splits with quite
different shares of its 478 positives.

If `data/processed/splits.npz` already exists it is reused, so this notebook and the rest of
the project report numbers on **identical** splits.

In [ ]:
SPLITS_NPZ = PROCESSED / "splits.npz"

def split_by_label_combination(Y, seed=SEED):
    """70/15/15 split stratified on the exact label COMBINATION of each row.

    Self-contained fallback for when `iterative-stratification` is unavailable, which is the
    normal case on Kaggle with internet off. With only 6 binary labels there are at most 64
    combinations, so stratifying on the combination itself is a finer partition than
    per-label stratification and preserves every label's rate by construction. Combinations
    too rare to divide three ways are pooled into one bucket and split randomly.
    """
    from sklearn.model_selection import train_test_split

    combo = (Y * (2 ** np.arange(Y.shape[1]))).sum(axis=1)   # unique id per label combination
    counts = pd.Series(combo).value_counts()
    rare = set(counts[counts < 6].index)                      # need >=6 to give each split >=1
    combo = np.where(np.isin(combo, list(rare)), -1, combo) if rare else combo

    idx = np.arange(len(Y))
    train_idx, holdout = train_test_split(idx, test_size=0.30, random_state=seed, stratify=combo)
    val_idx, test_idx = train_test_split(
        holdout, test_size=0.50, random_state=seed, stratify=combo[holdout]
    )
    return {"train": np.sort(train_idx), "val": np.sort(val_idx), "test": np.sort(test_idx)}


if SPLITS_NPZ.exists():
    with np.load(SPLITS_NPZ) as f:
        splits = {k: f[k] for k in ("train", "val", "test")}
    log.info("reused existing splits from %s", SPLITS_NPZ)
else:
    # Always the combination splitter -- deliberately NOT preferring
    # iterative-stratification when it happens to be installed. Preferring it would mean this
    # notebook produced one split locally and a different one on Kaggle, making the two runs'
    # metrics silently incomparable. src.data uses strategy="combination" for the same reason,
    # so notebook, CLI, and Kaggle all agree.
    splits = split_by_label_combination(Y)
    np.savez_compressed(SPLITS_NPZ, **splits)
    log.info("created splits (label-combination stratified) -> %s", SPLITS_NPZ)

assert len(set(splits["train"]) & set(splits["val"])) == 0
assert len(set(splits["train"]) & set(splits["test"])) == 0
assert len(set(splits["val"]) & set(splits["test"])) == 0

if QUICK_TEST:
    rng = np.random.default_rng(SEED)
    splits = {k: rng.choice(v, size=min(len(v), 2000 if k == "train" else 1000), replace=False) for k, v in splits.items()}
    log.warning("QUICK_TEST: subsampled to %s", {k: len(v) for k, v in splits.items()})

data = {k: (texts[i].tolist(), Y[i]) for k, i in splits.items()}
pd.DataFrame({k: dict(zip(LABELS, y.sum(0))) | {"n": len(y)} for k, (_, y) in data.items()})

## 4. Tokenize

Truncate at `MAX_LENGTH` but **do not pad here** — padding is applied per batch in section 6,
which is where the speed comes from.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# What does truncation actually cost? Worth knowing rather than assuming.
sample = data["train"][0][:20000]
lens = np.array([len(tokenizer.encode(t, truncation=False)) for t in sample])
print(f"token length  p50={np.percentile(lens,50):.0f}  p90={np.percentile(lens,90):.0f}  "
      f"p95={np.percentile(lens,95):.0f}  p99={np.percentile(lens,99):.0f}  max={lens.max()}")
for limit in (128, 256, 512):
    print(f"  max_length={limit:<4} truncates {(lens>limit).mean():6.2%} of comments, "
          f"losing {np.clip(lens-limit,0,None).sum()/lens.sum():6.2%} of all tokens")

encodings = {
    k: tokenizer(t, truncation=True, padding=False, max_length=MAX_LENGTH)
    for k, (t, _) in data.items()
}
log.info("tokenized: %s", {k: len(v["input_ids"]) for k, v in encodings.items()})

## 5. The model — six independent sigmoids, BCE loss

**This is the core multi-label concept.** The output layer produces 6 logits, each passed
through its *own* sigmoid, and the loss is binary cross-entropy applied per label.

It is **not** softmax + categorical cross-entropy:

- **Softmax couples the outputs.** It divides by the sum over all classes, so the six
  probabilities must total 1.0. Pushing one up pushes the others down. That encodes
  "exactly one of these is correct" — a single-winner competition.
- **Sigmoid treats each logit alone.** `σ(zᵢ) = 1/(1+e^{-zᵢ})` depends only on `zᵢ`, so the
  six outputs are free to be independently high or low. They can sum to 0.0 (a clean
  comment — 89.8% of this data) or to 6.0 (31 Jigsaw comments carry all six labels).

Concretely: a comment labeled `toxic + obscene + insult` is **one** example with **three**
simultaneously correct answers. Softmax cannot represent that target — it would split
probability mass three ways, driving each toward 0.33, and it could not represent the
all-zero case at all. Sigmoid + BCE asks six independent yes/no questions.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=N_LABELS,
    problem_type="multi_label_classification",   # makes HF use BCEWithLogitsLoss
    id2label=dict(enumerate(LABELS)),
    label2id={l: i for i, l in enumerate(LABELS)},
)

# Demonstration of the point above, on the untrained head.
with torch.no_grad():
    logits = model(**tokenizer("you are an idiot", return_tensors="pt")).logits

sig = torch.sigmoid(logits)[0]
soft = torch.softmax(logits, dim=-1)[0]
print(pd.DataFrame({"logit": logits[0], "sigmoid (used)": sig, "softmax (wrong here)": soft}, index=LABELS).round(4))
print(f"\nsigmoid sums to {sig.sum():.3f} -- unconstrained, each label independent")
print(f"softmax sums to {soft.sum():.3f} -- forced to 1.0, so labels compete for mass")

## 6. Dataset, dynamic padding, and the loss

The collator pads each batch to its own longest sequence instead of a fixed 256, and
`MultiLabelTrainer` groups similar-length examples into the same batch. Together these were
measured at **1.25 s/step → ~0.35 s/step**, i.e. ~2h25m → ~45 min for this run. The median
comment is 52 tokens against a 256 limit, so fixed padding wastes most of the compute.

There are three cells below. The first two define classes and run instantly. The third only
*measures* padding efficiency to illustrate the point — **it is safe to skip**; nothing in
training depends on it.

In [ ]:
class MultiLabelDataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = None if labels is None else labels.astype(np.float32)

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}   # variable-length lists
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item


class MultiLabelCollator:
    """Pads input_ids per batch; stacks the (batch, 6) float targets separately.

    Written by hand rather than using DataCollatorWithPadding because the labels here are a
    fixed-width float vector per example, not one class index, and must not be run through
    the tokenizer's padding logic.
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        features = [dict(f) for f in features]
        labels = [f.pop("labels") for f in features] if "labels" in features[0] else None
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        if labels is not None:
            batch["labels"] = torch.tensor(np.asarray(labels), dtype=torch.float32)
        return batch


datasets = {k: MultiLabelDataset(encodings[k], data[k][1]) for k in data}
collator = MultiLabelCollator(tokenizer)
print("datasets built:", {k: len(v) for k, v in datasets.items()}, flush=True)

b = collator([datasets["train"][i] for i in range(BATCH_SIZE)])
print("one batch:", {k: tuple(v.shape) for k, v in b.items()}, flush=True)
print(f"labels are (batch, {N_LABELS}) floats -- one 0/1 target per label, not a class index", flush=True)

In [ ]:
from sklearn.metrics import average_precision_score, f1_score, precision_recall_fscore_support
from transformers import Trainer, TrainingArguments

try:
    from transformers.trainer_pt_utils import LengthGroupedSampler
except ImportError:  # moved or removed in some version -- fall back to default shuffling
    LengthGroupedSampler = None


def evaluate(y_true, y_prob, thresholds=0.5, name="model"):
    """Macro-F1 is the headline. Micro-F1 is reported alongside because the GAP between them
    measures rare-label neglect: micro pools every decision (so `toxic`'s 2,294 positives
    outvote `threat`'s 72 by ~32:1), macro weights all six labels equally."""
    y_true = np.asarray(y_true)
    t = np.full(N_LABELS, float(thresholds)) if np.isscalar(thresholds) else np.array([thresholds[l] for l in LABELS])
    y_pred = (y_prob >= t[None, :]).astype(np.int8)

    p, r, f1, sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=range(N_LABELS))
    ap = [average_precision_score(y_true[:, i], y_prob[:, i]) if y_true[:, i].any() else float("nan") for i in range(N_LABELS)]

    macro = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    micro = float(f1_score(y_true, y_pred, average="micro", zero_division=0))
    return {
        "name": name,
        "macro_f1": macro,                      # <- headline
        "micro_f1": micro,
        "micro_macro_gap": micro - macro,
        "macro_average_precision": float(np.nanmean(ap)),
        "per_label": {
            l: {"threshold": float(t[i]), "precision": float(p[i]), "recall": float(r[i]),
                "f1": float(f1[i]), "average_precision": float(ap[i]), "support": int(sup[i]),
                "n_predicted_positive": int(y_pred[:, i].sum())}
            for i, l in enumerate(LABELS)
        },
        # Included ONLY to show it is misleading: an all-zeros model gets these same numbers
        # with macro-F1 = 0.0.
        "accuracy_do_not_use_as_headline": {
            "exact_match": float((y_pred == y_true).all(1).mean()),
            "_caveat": "an all-zeros model scores ~89.8% here at macro-F1 0.0",
        },
    }


def report(m):
    print(f"=== {m['name']} ===")
    print(f"  MACRO-F1 (headline) : {m['macro_f1']:.4f}")
    print(f"  micro-F1            : {m['micro_f1']:.4f}  (gap {m['micro_macro_gap']:+.4f})")
    print(f"  macro-AP            : {m['macro_average_precision']:.4f}")
    print(f"  {'label':<14} {'P':>6} {'R':>6} {'F1':>6} {'AP':>6} {'support':>8} {'pred':>6}")
    for l, d in m["per_label"].items():
        print(f"  {l:<14} {d['precision']:6.3f} {d['recall']:6.3f} {d['f1']:6.3f} "
              f"{d['average_precision']:6.3f} {d['support']:8d} {d['n_predicted_positive']:6d}")


class MultiLabelTrainer(Trainer):
    def _get_train_sampler(self, train_dataset=None):
        """Batch examples of similar length together.

        This is the other half of the speedup. A random batch of 32 almost always
        contains one long comment (P(any > 200 tokens) ~ 97%), so with dynamic padding
        alone nearly every batch would still pad out to ~256.

        Done by overriding the sampler because the `group_by_length` TrainingArguments
        flag was removed in transformers v5. LengthGroupedSampler itself still exists,
        and passing the lengths in explicitly is clearer than letting Trainer infer them.
        """
        ds = train_dataset if train_dataset is not None else self.train_dataset
        if ds is None or LengthGroupedSampler is None:
            return super()._get_train_sampler(train_dataset) if train_dataset is not None else super()._get_train_sampler()
        g = torch.Generator()
        g.manual_seed(self.args.seed)
        return LengthGroupedSampler(
            batch_size=self.args.train_batch_size,
            lengths=[len(ids) for ids in ds.enc["input_ids"]],
            generator=g,
        )

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        # BCEWithLogitsLoss = sigmoid + binary cross-entropy, fused for numerical stability.
        # Applied elementwise: each of the 6 logits is scored against its own 0/1 target,
        # with NO normalization across the label dimension. That absence of normalization is
        # exactly what separates this from softmax + cross-entropy.
        loss = nn.BCEWithLogitsLoss()(outputs.logits, labels.float())
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    probs = torch.sigmoid(torch.as_tensor(eval_pred.predictions)).numpy()
    m = evaluate(eval_pred.label_ids.astype(np.int8), probs, 0.5)
    return {"macro_f1": m["macro_f1"], "micro_f1": m["micro_f1"],
            "macro_ap": m["macro_average_precision"], "f1_threat": m["per_label"]["threat"]["f1"]}


# This cell only defines things, so say so explicitly -- a cell that finishes instantly with
# no output at all is impossible to tell apart from one that is hung.
print("defined: evaluate(), report(), MultiLabelTrainer, compute_metrics -- nothing to compute here", flush=True)

In [ ]:
# ILLUSTRATION ONLY -- nothing below is needed for training, so it is safe to skip.
# Measured on a 20k sample rather than all 111,699 examples to keep this instant.
#
# The point: dynamic padding ALONE barely helps here, because a random batch of 32 nearly
# always contains one long comment and pads out to the limit anyway. Grouping similar-length
# comments together is what actually cuts the wasted compute.
DEMO_N = 20_000
all_lengths = np.array([len(x) for x in datasets["train"].enc["input_ids"]])
lengths = all_lengths[:DEMO_N].tolist()
print(f"measuring padding efficiency on {len(lengths):,} of {len(all_lengths):,} examples...", flush=True)

rng_demo = np.random.default_rng(SEED)
random_widths = [
    max(lengths[int(i)] for i in rng_demo.integers(0, len(lengths), BATCH_SIZE))
    for _ in range(50)
]

if LengthGroupedSampler is not None:
    g = torch.Generator()
    g.manual_seed(SEED)
    order = list(LengthGroupedSampler(batch_size=BATCH_SIZE, lengths=lengths, generator=g))
else:  # approximate the same effect for the illustration
    order = list(np.argsort(lengths))
grouped_widths = [
    max(lengths[i] for i in order[s : s + BATCH_SIZE])
    for s in range(0, len(order) - BATCH_SIZE + 1, BATCH_SIZE)
]

print(f"mean tokens per example (no padding at all) : {all_lengths.mean():6.1f}")
print(f"fixed padding to MAX_LENGTH                 : {MAX_LENGTH:6.1f}  <- wastes the most")
print(f"dynamic padding, random batches             : {np.mean(random_widths):6.1f}")
print(f"dynamic padding + length grouping           : {np.mean(grouped_widths):6.1f}  <- what we use")
print(f"\nspeedup vs fixed padding: ~{MAX_LENGTH / np.mean(grouped_widths):.1f}x less compute per step")

## 7. Train

Model selection uses **macro-F1** on validation (`metric_for_best_model`), not loss and not
accuracy — the metric we actually care about is the one that picks the checkpoint.

In [ ]:
import inspect

import transformers

# Kaggle pins its own transformers version, and the Trainer API differs across majors:
#   - `evaluation_strategy` was renamed `eval_strategy` (4.41+)
#   - `tokenizer=` was renamed `processing_class=` (4.46+)
#   - `group_by_length` was REMOVED in v5 (handled by MultiLabelTrainer's sampler override)
# Rather than pin a version, ask the installed classes what they accept.
TA_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)
TRAINER_PARAMS = set(inspect.signature(Trainer.__init__).parameters)
print(f"transformers {transformers.__version__}", flush=True)

steps_per_epoch = -(-len(datasets["train"]) // BATCH_SIZE)
OUT_DIR = MODELS / "distilbert-multilabel"

wanted = dict(
    output_dir=str(OUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=int(WARMUP_RATIO * steps_per_epoch * EPOCHS),
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=100,
    seed=SEED,
    data_seed=SEED,
    fp16=(DEVICE == "cuda"),   # big speedup on a Kaggle GPU; unsupported by Trainer on MPS
    bf16=False,
    dataloader_num_workers=0,
    report_to=[],
)
# Whichever name this version uses for the eval-strategy argument.
wanted["eval_strategy" if "eval_strategy" in TA_PARAMS else "evaluation_strategy"] = "epoch"

dropped = sorted(set(wanted) - TA_PARAMS)
if dropped:
    print(f"note: this transformers version does not accept {dropped} -- skipping", flush=True)
args = TrainingArguments(**{k: v for k, v in wanted.items() if k in TA_PARAMS})

trainer_kwargs = dict(
    model=model,
    args=args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["val"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer_kwargs["processing_class" if "processing_class" in TRAINER_PARAMS else "tokenizer"] = tokenizer

trainer = MultiLabelTrainer(**trainer_kwargs)
print("trainer ready -- starting training", flush=True)

t0 = time.perf_counter()
train_result = trainer.train()
train_minutes = (time.perf_counter() - t0) / 60
print(f"\ntrained in {train_minutes:.1f} min | final loss {train_result.training_loss:.4f}")

trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))
print(f"saved model -> {OUT_DIR}")

## 8. Predict and evaluate at the default 0.5

Probabilities for **both** val and test are saved to disk: threshold tuning in section 9
needs val, and the frozen thresholds are then applied to test.

In [ ]:
@torch.no_grad()
def predict_probs(model, ds, batch_size=EVAL_BATCH_SIZE):
    """Sigmoid probabilities, shape (n, 6). Runs in length-sorted order so batches pad
    tightly, then scatters results back to the original row order."""
    model.eval()
    lengths = np.array([len(ds.enc["input_ids"][i]) for i in range(len(ds))])
    order = np.argsort(lengths, kind="stable")
    out = np.empty((len(ds), N_LABELS), dtype=np.float32)
    for s in range(0, len(order), batch_size):
        idx = order[s : s + batch_size]
        batch = collator([ds[int(i)] for i in idx])
        batch.pop("labels", None)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out[idx] = torch.sigmoid(model(**batch).logits).float().cpu().numpy()
    return out


probs, metrics_at_half = {}, {}
for split in ("val", "test"):
    t0 = time.perf_counter()
    probs[split] = predict_probs(model, datasets[split])
    np.save(PROBS / f"transformer_{split}.npy", probs[split])
    metrics_at_half[split] = evaluate(data[split][1], probs[split], 0.5, name=f"DistilBERT @0.5 ({split})")
    print(f"[{split}] {len(probs[split]):,} rows in {(time.perf_counter()-t0)/60:.1f} min "
          f"-> {PROBS / f'transformer_{split}.npy'}")

report(metrics_at_half["test"])

## 9. Per-label threshold calibration — the biggest win, for free

The model gives probabilities; **you** choose the operating point. 0.5 is arbitrary.

A classifier trained on a label with 0.30% prevalence sees 333 negatives per positive, so its
posterior for a genuine threat can sit at 0.2 and still mean "far more likely than a random
comment". Cutting at 0.5 throws that detection away.

**Protocol, which is the part that matters:** thresholds are chosen on **validation**, then
frozen and applied to **test**. Tuning on test would let the test labels pick the operating
point and report an optimistically biased number.

The grid is deliberately coarse (19 points). `threat` has only ~71 validation positives, so a
finer grid would fit noise — the validation-to-test gap printed below quantifies that cost.

In [ ]:
GRID = np.linspace(0.05, 0.95, 19)
Y_val, Y_test = data["val"][1], data["test"][1]

thresholds, diagnostics = {}, {}
for i, label in enumerate(LABELS):
    # VALIDATION ONLY in this loop.
    f1s = np.array([f1_score(Y_val[:, i], probs["val"][:, i] >= t, zero_division=0) for t in GRID])
    best = int(np.argmax(f1s))
    thresholds[label] = float(GRID[best])
    diagnostics[label] = {
        "chosen_threshold": float(GRID[best]),
        "val_f1_at_chosen": float(f1s[best]),
        "val_f1_at_0.5": float(f1_score(Y_val[:, i], probs["val"][:, i] >= 0.5, zero_division=0)),
        "val_positives": int(Y_val[:, i].sum()),
        "f1_curve": {f"{t:.2f}": float(f) for t, f in zip(GRID, f1s)},
    }

# Thresholds are now frozen. Only now do we touch test.
before = metrics_at_half["test"]
after = evaluate(Y_test, probs["test"], thresholds, name="DistilBERT @tuned (test)")

comparison = pd.DataFrame({
    "threshold": [thresholds[l] for l in LABELS],
    "F1 @0.5": [before["per_label"][l]["f1"] for l in LABELS],
    "F1 tuned": [after["per_label"][l]["f1"] for l in LABELS],
    "gain": [after["per_label"][l]["f1"] - before["per_label"][l]["f1"] for l in LABELS],
    "recall @0.5": [before["per_label"][l]["recall"] for l in LABELS],
    "recall tuned": [after["per_label"][l]["recall"] for l in LABELS],
    "val positives": [diagnostics[l]["val_positives"] for l in LABELS],
    "val->test gap": [after["per_label"][l]["f1"] - diagnostics[l]["val_f1_at_chosen"] for l in LABELS],
}, index=LABELS).round(4)
display(comparison)

gain = after["macro_f1"] - before["macro_f1"]
# Guard the division: a model that predicts nothing at 0.5 has macro-F1 exactly 0.0, which
# is not a hypothetical -- it is the failure mode this whole project is about, and it happens
# for real on rare labels and on undertrained models.
rel = f"{100 * gain / before['macro_f1']:+.1f}%" if before["macro_f1"] > 0 else "from zero"
print(f"MACRO-F1  {before['macro_f1']:.4f} (@0.5)  ->  {after['macro_f1']:.4f} (tuned)   {gain:+.4f}  ({rel})")

worst = comparison["val->test gap"].idxmin()
print(f"\nCaveat: `{worst}` drops {comparison.loc[worst, 'val->test gap']:+.4f} from validation to test, "
      f"tuned on only {diagnostics[worst]['val_positives']} validation positives.")
print("Thresholds for rare labels are estimated from very few positives, so their validation")
print("F1 is optimistic. A real limitation of the method -- reported, not hidden.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))

for label in LABELS:
    d = diagnostics[label]
    ts = np.array([float(t) for t in d["f1_curve"]])
    axes[0].plot(ts, list(d["f1_curve"].values()), marker="o", ms=3, label=f"{label} (t*={d['chosen_threshold']:.2f})")
    axes[0].scatter([d["chosen_threshold"]], [d["val_f1_at_chosen"]], s=80, zorder=5, edgecolor="black", linewidth=0.7)
axes[0].axvline(0.5, ls="--", color="grey", lw=1.2)
axes[0].text(0.51, 0.02, "default 0.5", fontsize=9, color="grey")
axes[0].set(xlabel="threshold", ylabel="F1 on validation", title="F1 vs threshold — the optimum is label-specific")
axes[0].legend(fontsize=8, ncol=2)
axes[0].grid(alpha=0.3)

x = np.arange(len(LABELS))
axes[1].bar(x - 0.2, comparison["F1 @0.5"], 0.4, label="F1 @ 0.5", color="#4C72B0")
axes[1].bar(x + 0.2, comparison["F1 tuned"], 0.4, label="F1 tuned", color="#C44E52")
axes[1].set_xticks(x, LABELS, rotation=25, ha="right")
axes[1].set(ylabel="F1 on test", title=f"macro-F1 {before['macro_f1']:.3f} → {after['macro_f1']:.3f}")
axes[1].legend()
plt.tight_layout()
plt.savefig(RESULTS / "threshold_sweep_transformer.png", bbox_inches="tight", dpi=110)
plt.show()

## 10. Save everything

Written to the paths the other notebooks and `src/` modules read, so the error analysis and
comparison phases pick these up with no further work.

In [ ]:
(RESULTS / "transformer_metrics.json").write_text(json.dumps({
    "phase": 3,
    "model": MODEL_NAME,
    "loss": "BCEWithLogitsLoss (independent sigmoid per label, not softmax)",
    "hyperparameters": {
        "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS, "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO,
        "seed": SEED, "quick_test": QUICK_TEST,
    },
    "device": DEVICE,
    "train_minutes": train_minutes,
    "train_loss": float(train_result.training_loss),
    "n_train": len(datasets["train"]),
    "splits": {"val": metrics_at_half["val"], "test": metrics_at_half["test"]},
}, indent=2))

(RESULTS / "thresholds_transformer.json").write_text(json.dumps({
    "model": "transformer",
    "grid": GRID.tolist(),
    "tuned_on": "validation split only — test was not consulted during selection",
    "thresholds": thresholds,
    "per_label": diagnostics,
}, indent=2))

(RESULTS / "threshold_comparison_transformer.json").write_text(json.dumps({
    "model": "transformer",
    "headline": {
        "macro_f1_at_0.5": before["macro_f1"],
        "macro_f1_tuned": after["macro_f1"],
        "macro_f1_gain": gain,
        "relative_gain_percent": (100 * gain / before["macro_f1"]) if before["macro_f1"] > 0 else None,
    },
    "micro_f1_at_0.5": before["micro_f1"],
    "micro_f1_tuned": after["micro_f1"],
    "per_label": {
        l: {
            "threshold": thresholds[l],
            "test_f1_at_0.5": before["per_label"][l]["f1"],
            "test_f1_tuned": after["per_label"][l]["f1"],
            "test_f1_gain": after["per_label"][l]["f1"] - before["per_label"][l]["f1"],
            "val_f1_tuned": diagnostics[l]["val_f1_at_chosen"],
            "val_to_test_gap": after["per_label"][l]["f1"] - diagnostics[l]["val_f1_at_chosen"],
            "val_positives": diagnostics[l]["val_positives"],
            "test_positives": before["per_label"][l]["support"],
            "recall_at_0.5": before["per_label"][l]["recall"],
            "recall_tuned": after["per_label"][l]["recall"],
            "precision_at_0.5": before["per_label"][l]["precision"],
            "precision_tuned": after["per_label"][l]["precision"],
        }
        for l in LABELS
    },
}, indent=2))

(RESULTS / "transformer_tuned_metrics.json").write_text(json.dumps(after, indent=2))

for p in ["transformer_metrics.json", "thresholds_transformer.json",
          "threshold_comparison_transformer.json", "transformer_tuned_metrics.json"]:
    print("wrote", RESULTS / p)
print("wrote", PROBS / "transformer_val.npy")
print("wrote", PROBS / "transformer_test.npy")

if QUICK_TEST:
    print("\n*** QUICK_TEST was True -- these numbers are from a tiny subsample. "
          "Set QUICK_TEST = False and re-run for real results. ***")

## Summary

| | |
|---|---|
| **Headline** | macro-F1 on test, with per-label thresholds tuned on validation |
| **Never used** | accuracy — an all-zeros model scores ~89.8% exact-match at macro-F1 0.0 |
| **Free win** | per-label thresholds, no retraining |
| **Honest caveat** | `threat`'s threshold rests on ~71 validation positives, so its tuned score is noisy |

**What to run next:** `notebooks/03_error_analysis.ipynb` reads the probability files saved
above and digs into *what* the model gets wrong — which labels it conflates, whether it
reproduces the real label co-occurrence structure, and a set of hand-inspected failures.